In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# Custom Native Geohash Decoder
def decode_geohash(geohash):
    __base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    __decodemap = {__base32[i]: i for i in range(len(__base32))}
    lat_interval, lon_interval = (-90.0, 90.0), (-180.0, 180.0)
    is_even = True
    for c in geohash:
        cd = __decodemap[c]
        for mask in [16, 8, 4, 2, 1]:
            if is_even:
                if cd & mask:
                    lon_interval = ((lon_interval[0]+lon_interval[1])/2, lon_interval[1])
                else:
                    lon_interval = (lon_interval[0], (lon_interval[0]+lon_interval[1])/2)
            else:
                if cd & mask:
                    lat_interval = ((lat_interval[0]+lat_interval[1])/2, lat_interval[1])
                else:
                    lat_interval = (lat_interval[0], (lat_interval[0]+lat_interval[1])/2)
            is_even = not is_even
    return (lat_interval[0] + lat_interval[1]) / 2, (lon_interval[0] + lon_interval[1]) / 2

# Load Data
print("Loading data...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train['is_train'] = 1
test['is_train'] = 0
test['demand'] = np.nan
df = pd.concat([train, test], ignore_index=True)

# FEATURE ENGINEERING
print("Extracting Granular Time & Geo-Stats Features...")
df[['hour', 'minute']] = df['timestamp'].str.split(':', expand=True).astype(int)

# 96-Slot Day Upgrade & Cyclical Math
df['time_slot'] = df['hour'] * 4 + df['minute'] // 15
df['slot_sin'] = np.sin(2 * np.pi * df['time_slot'] / 96.0)
df['slot_cos'] = np.cos(2 * np.pi * df['time_slot'] / 96.0)

# Spatial Decoding
decoded = df['geohash'].apply(decode_geohash)
df['latitude'] = [x[0] for x in decoded]
df['longitude'] = [x[1] for x in decoded]

# Data Imputation
df['Temperature'] = df.groupby('geohash')['Temperature'].transform(lambda x: x.ffill().bfill())
df['Weather'] = df.groupby('geohash')['Weather'].transform(lambda x: x.ffill().bfill())
roadtype_modes = df.groupby('geohash')['RoadType'].agg(lambda x: x.mode()[0] if not x.mode().empty else 'Unknown')
df['RoadType'] = df['RoadType'].fillna(df['geohash'].map(roadtype_modes))
df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())
df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0])
df['loc_slot'] = df['geohash'] + "_" + df['time_slot'].astype(str)

# Categorical Setup
cat_cols = ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks', 'geohash']
for col in cat_cols:
    df[col] = df[col].astype('category')

# Split Back into Train/Test
train_clean = df[df['is_train'] == 1].drop(['is_train', 'timestamp'], axis=1)
test_clean = df[df['is_train'] == 0].drop(['is_train', 'timestamp', 'demand'], axis=1)

features = [c for c in train_clean.columns if c not in ['Index', 'loc_slot', 'demand']]

# SMOOTHED K-FOLD TARGET ENCODING
print("Calculating Smoothed Geo-Statistical Target Encodings...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_clean))
test_preds = np.zeros(len(test_clean))

for col in ['loc_slot_enc', 'geo_mean_demand', 'geo_std_demand']:
    train_clean[col] = 0.0
    test_clean[col] = 0.0

global_mean = train_clean['demand'].mean()
global_std = train_clean['demand'].std()
ALPHA = 20

for train_idx, val_idx in kf.split(train_clean):
    X_tr, X_va = train_clean.iloc[train_idx], train_clean.iloc[val_idx]

    # Smoothed loc_slot
    loc_slot_stats = X_tr.groupby('loc_slot')['demand'].agg(['mean', 'count'])
    loc_slot_smooth = (loc_slot_stats['count'] * loc_slot_stats['mean'] + ALPHA * global_mean) / (loc_slot_stats['count'] + ALPHA)

    # Smoothed geohash
    geo_stats = X_tr.groupby('geohash')['demand'].agg(['mean', 'count', 'std'])
    geo_mean_smooth = (geo_stats['count'] * geo_stats['mean'] + ALPHA * global_mean) / (geo_stats['count'] + ALPHA)

    train_clean.loc[val_idx, 'loc_slot_enc'] = X_va['loc_slot'].map(loc_slot_smooth.to_dict()).fillna(global_mean)
    train_clean.loc[val_idx, 'geo_mean_demand'] = X_va['geohash'].map(geo_mean_smooth.to_dict()).fillna(global_mean)
    train_clean.loc[val_idx, 'geo_std_demand'] = X_va['geohash'].map(geo_stats['std'].to_dict()).fillna(global_std)

# Encode Official Test Set
full_loc_slot_stats = train_clean.groupby('loc_slot')['demand'].agg(['mean', 'count'])
full_loc_slot_smooth = (full_loc_slot_stats['count'] * full_loc_slot_stats['mean'] + ALPHA * global_mean) / (full_loc_slot_stats['count'] + ALPHA)

full_geo_stats = train_clean.groupby('geohash')['demand'].agg(['mean', 'count', 'std'])
full_geo_mean_smooth = (full_geo_stats['count'] * full_geo_stats['mean'] + ALPHA * global_mean) / (full_geo_stats['count'] + ALPHA)

test_clean['loc_slot_enc'] = test_clean['loc_slot'].map(full_loc_slot_smooth.to_dict()).fillna(global_mean)
test_clean['geo_mean_demand'] = test_clean['geohash'].map(full_geo_mean_smooth.to_dict()).fillna(global_mean)
test_clean['geo_std_demand'] = test_clean['geohash'].map(full_geo_stats['std'].to_dict()).fillna(global_std)

features.extend(['loc_slot_enc', 'geo_mean_demand', 'geo_std_demand'])

X_train = train_clean[features]
y_train_raw = train_clean['demand']
y_train_log = np.log1p(train_clean['demand'])
X_test = test_clean[features]


# TRI-CORE ENSEMBLE TRAINING (LGBM_Deep + LGBM_Shallow + CatBoost)
print("\nTraining 5-Fold Tri-Core Ensemble...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train_log)):
    print(f"\n--> Training Fold {fold + 1}/5...")
    X_tr, y_tr_log = X_train.iloc[train_idx], y_train_log.iloc[train_idx]
    X_va, y_va_log = X_train.iloc[val_idx], y_train_log.iloc[val_idx]

    # MODEL 1: LightGBM (Deep)
    lgb_deep = lgb.LGBMRegressor(
        n_estimators=1500, learning_rate=0.03, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, random_state=42 + fold, n_jobs=-1
    )
    lgb_deep.fit(X_tr, y_tr_log, eval_set=[(X_va, y_va_log)], callbacks=[lgb.early_stopping(50, verbose=False)])

    # MODEL 2: LightGBM (Shallow)
    lgb_shallow = lgb.LGBMRegressor(
        n_estimators=1500, learning_rate=0.03, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, random_state=123 + fold, n_jobs=-1
    )
    lgb_shallow.fit(X_tr, y_tr_log, eval_set=[(X_va, y_va_log)], callbacks=[lgb.early_stopping(50, verbose=False)])

    # MODEL 3: CatBoost (Categorical)
    X_tr_cb, X_va_cb, X_test_cb = X_tr.copy(), X_va.copy(), X_test.copy()
    for c in cat_cols:
        X_tr_cb[c], X_va_cb[c], X_test_cb[c] = X_tr_cb[c].astype(str), X_va_cb[c].astype(str), X_test_cb[c].astype(str)

    cb_model = CatBoostRegressor(
        iterations=1500, learning_rate=0.04, depth=8,
        cat_features=cat_cols, random_seed=42 + fold,
        eval_metric='RMSE', early_stopping_rounds=50, verbose=False
    )
    cb_model.fit(X_tr_cb, y_tr_log, eval_set=(X_va_cb, y_va_log), use_best_model=True)

    # Ensemble Modelling (40+20+40)
    pred_va = (np.expm1(lgb_deep.predict(X_va)) * 0.40) + \
              (np.expm1(lgb_shallow.predict(X_va)) * 0.20) + \
              (np.expm1(cb_model.predict(X_va_cb)) * 0.40)

    pred_te = (np.expm1(lgb_deep.predict(X_test)) * 0.40) + \
              (np.expm1(lgb_shallow.predict(X_test)) * 0.20) + \
              (np.expm1(cb_model.predict(X_test_cb)) * 0.40)

    oof_preds[val_idx] = pred_va
    test_preds += pred_te / kf.n_splits

# Final Evaluation
final_r2 = r2_score(y_train_raw, oof_preds)
print(f"\n------------------------------------")
print(f"TRI-CORE OOF R2 SCORE: {final_r2:.4f}")
print(f"HACKATHON ESTIMATE:    {max(0, 100 * final_r2):.2f} / 100")
print(f"--------------------------------------")

test_preds = np.clip(test_preds, 0, 1)
submission = pd.DataFrame({'Index': test_clean['Index'], 'demand': test_preds})
submission.to_csv('v1_9_tricore_ensemble_model.csv', index=False)
print("Saved v1_9_tricore_ensemble_model.csv! The model is ready.")

Loading data...
Extracting Granular Time & Geo-Stats Features...
Calculating Smoothed Geo-Statistical Target Encodings...

Training 5-Fold Tri-Core Ensemble...

--> Training Fold 1/5...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009627 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2465
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 18
[LightGBM] [Info] Start training from score 0.083012
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM]